[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/03_minimal_ai_systems_overview.ipynb)

# 03. Minimal AI systems overview — architecture → training → inference

이 노트북 하나만 열어도 모델 구조를 직접 읽고 실행할 수 있게 한다. 각 모델은 **architecture/forward → tiny training → inference/sampling → 구조 검증** 순서로 본다.

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)
print("torch:", torch.__version__)

## 0. 공통 Transformer 부품

attention 계산을 `nn.TransformerEncoderLayer`에 숨기지 않고 직접 구현한다.

`X → Q,K,V → QKᵀ/√d_h → mask → softmax → V weighted sum → output projection`

In [ ]:
def apply_rope(q, k):
    head_dim = q.size(-1)
    assert head_dim % 2 == 0

    positions = torch.arange(
        q.size(-2),
        device=q.device,
        dtype=q.dtype,
    )
    frequencies = torch.arange(
        0,
        head_dim,
        2,
        device=q.device,
        dtype=q.dtype,
    )
    inverse_frequencies = 1.0 / (
        10000 ** (frequencies / head_dim)
    )

    angles = positions[:, None] * inverse_frequencies[None, :]
    cos = angles.cos()[None, None]
    sin = angles.sin()[None, None]

    def rotate(x):
        even = x[..., 0::2]
        odd = x[..., 1::2]
        rotated = torch.stack(
            (
                even * cos - odd * sin,
                even * sin + odd * cos,
            ),
            dim=-1,
        )
        return rotated.flatten(-2)

    return rotate(q), rotate(k)


class SelfAttention(nn.Module):
    def __init__(
        self,
        hidden_dim,
        num_heads,
        use_rope=False,
    ):
        super().__init__()
        assert hidden_dim % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.use_rope = use_rope

        self.qkv = nn.Linear(
            hidden_dim,
            3 * hidden_dim,
        )
        self.out_projection = nn.Linear(
            hidden_dim,
            hidden_dim,
        )

    def forward(
        self,
        x,
        allowed_mask=None,
    ):
        batch_size, sequence_length, hidden_dim = x.shape

        qkv = self.qkv(x)
        qkv = qkv.reshape(
            batch_size,
            sequence_length,
            3,
            self.num_heads,
            self.head_dim,
        )
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        if self.use_rope:
            q, k = apply_rope(q, k)

        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)

        if allowed_mask is not None:
            if allowed_mask.ndim == 2:
                allowed_mask = allowed_mask[None, None]
            elif allowed_mask.ndim == 3:
                allowed_mask = allowed_mask[:, None]

            scores = scores.masked_fill(
                ~allowed_mask,
                torch.finfo(scores.dtype).min,
            )

        weights = scores.softmax(dim=-1)
        attended = weights @ v
        attended = attended.transpose(1, 2).contiguous()
        attended = attended.reshape(
            batch_size,
            sequence_length,
            hidden_dim,
        )

        output = self.out_projection(attended)
        return output, weights


class PreNormBlock(nn.Module):
    def __init__(
        self,
        hidden_dim,
        num_heads,
        mlp_ratio=4,
        use_rope=False,
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.attention = SelfAttention(
            hidden_dim,
            num_heads,
            use_rope=use_rope,
        )
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.mlp = nn.Sequential(
            nn.Linear(
                hidden_dim,
                mlp_ratio * hidden_dim,
            ),
            nn.GELU(),
            nn.Linear(
                mlp_ratio * hidden_dim,
                hidden_dim,
            ),
        )

    def forward(
        self,
        x,
        allowed_mask=None,
    ):
        attention_output, weights = self.attention(
            self.norm1(x),
            allowed_mask,
        )
        x = x + attention_output

        mlp_output = self.mlp(
            self.norm2(x)
        )
        x = x + mlp_output

        return x, weights

## A. Tiny GPT-2-like decoder

핵심 구조는 **learned token embedding + learned absolute position + causal self-attention + pre-norm residual + MLP + tied LM head**다.

In [ ]:
class TinyGPT(nn.Module):
    def __init__(
        self,
        vocab_size=32,
        max_length=16,
        hidden_dim=24,
        num_heads=3,
        depth=2,
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_dim,
        )
        self.position_embedding = nn.Embedding(
            max_length,
            hidden_dim,
        )
        self.blocks = nn.ModuleList(
            [
                PreNormBlock(
                    hidden_dim,
                    num_heads,
                )
                for _ in range(depth)
            ]
        )
        self.final_norm = nn.LayerNorm(hidden_dim)

    def forward(
        self,
        tokens,
        return_attn=False,
    ):
        sequence_length = tokens.size(1)
        positions = torch.arange(
            sequence_length,
            device=tokens.device,
        )

        x = self.token_embedding(tokens)
        x = x + self.position_embedding(positions)[None]

        causal_mask = torch.tril(
            torch.ones(
                sequence_length,
                sequence_length,
                dtype=torch.bool,
                device=tokens.device,
            )
        )

        attention_maps = []
        for block in self.blocks:
            x, weights = block(
                x,
                causal_mask,
            )
            attention_maps.append(weights)

        x = self.final_norm(x)
        logits = F.linear(
            x,
            self.token_embedding.weight,
        )

        if return_attn:
            return logits, attention_maps
        return logits


gpt = TinyGPT().to(device)
tokens = torch.tensor(
    [[1, 2, 3, 4, 5, 6, 7, 8]],
    device=device,
)
optimizer = torch.optim.AdamW(
    gpt.parameters(),
    lr=3e-3,
)

for step in range(4):
    input_tokens = tokens[:, :-1]
    target_tokens = tokens[:, 1:]
    logits = gpt(input_tokens)
    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        target_tokens.reshape(-1),
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(
        "GPT train",
        step,
        "loss",
        round(loss.item(), 4),
    )


@torch.no_grad()
def greedy_generate(
    model,
    prompt_tokens,
    new_tokens=4,
):
    generated = prompt_tokens.clone()

    for _ in range(new_tokens):
        logits = model(generated)
        next_token = logits[:, -1].argmax(
            dim=-1,
            keepdim=True,
        )
        generated = torch.cat(
            [generated, next_token],
            dim=1,
        )

    return generated


prompt = torch.tensor(
    [[1, 2, 3]],
    device=device,
)
generated = greedy_generate(
    gpt,
    prompt,
    new_tokens=4,
)
print("prompt:", prompt)
print("generated:", generated)

with torch.no_grad():
    _, attention_maps = gpt(
        tokens[:, :-1],
        return_attn=True,
    )

sequence_length = tokens.size(1) - 1
future_positions = torch.triu(
    torch.ones(
        sequence_length,
        sequence_length,
        dtype=torch.bool,
        device=device,
    ),
    diagonal=1,
)
future_mass = attention_maps[0][
    ...,
    future_positions,
].sum()
print(
    "future attention mass:",
    float(future_mass),
)

## B. Tiny ViT

`image → non-overlapping patches → patch projection → CLS token + learned position → full self-attention encoder → classifier`를 유지한다.

In [ ]:
class TinyViT(nn.Module):
    def __init__(
        self,
        image_size=16,
        patch_size=4,
        hidden_dim=24,
        num_heads=3,
        depth=2,
        num_classes=3,
    ):
        super().__init__()

        self.patch_size = patch_size
        num_patches = (image_size // patch_size) ** 2

        self.patch_projection = nn.Linear(
            3 * patch_size * patch_size,
            hidden_dim,
        )
        self.cls_token = nn.Parameter(
            torch.zeros(1, 1, hidden_dim)
        )
        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                1 + num_patches,
                hidden_dim,
            )
        )
        self.blocks = nn.ModuleList(
            [
                PreNormBlock(
                    hidden_dim,
                    num_heads,
                )
                for _ in range(depth)
            ]
        )
        self.final_norm = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Linear(
            hidden_dim,
            num_classes,
        )

        nn.init.normal_(self.cls_token, std=0.02)
        nn.init.normal_(self.position_embedding, std=0.02)

    def forward(
        self,
        image,
        return_attn=False,
    ):
        patches = F.unfold(
            image,
            kernel_size=self.patch_size,
            stride=self.patch_size,
        ).transpose(1, 2)

        x = self.patch_projection(patches)
        cls = self.cls_token.expand(
            image.size(0),
            -1,
            -1,
        )
        x = torch.cat([cls, x], dim=1)
        x = x + self.position_embedding[:, : x.size(1)]

        attention_maps = []
        for block in self.blocks:
            x, weights = block(x)
            attention_maps.append(weights)

        cls_hidden = self.final_norm(x)[:, 0]
        logits = self.classifier(cls_hidden)

        if return_attn:
            return logits, attention_maps
        return logits


vit = TinyViT().to(device)
images = torch.randn(
    4, 3, 16, 16,
    device=device,
)
labels = torch.tensor(
    [0, 1, 2, 1],
    device=device,
)
optimizer = torch.optim.AdamW(
    vit.parameters(),
    lr=3e-3,
)

for step in range(4):
    logits = vit(images)
    loss = F.cross_entropy(
        logits,
        labels,
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(
        "ViT train",
        step,
        "loss",
        round(loss.item(), 4),
    )

with torch.no_grad():
    logits, attention_maps = vit(
        images,
        return_attn=True,
    )
    predictions = logits.argmax(dim=-1)

print("ViT predictions:", predictions)
print(
    "attention shape:",
    tuple(attention_maps[0].shape),
)

## C. Tiny 2D/3D DiT + Flow Matching

DiT의 **patch token + fixed sin-cos position + timestep embedding + adaLN-Zero + conditioned final layer + unpatchify**를 유지한다. 학습 목표는 Flow Matching의 velocity regression이고, 추론에서는 noise에서 시작해 velocity field를 적분한다.

In [ ]:
def sinusoidal_1d(positions, embedding_dim):
    assert embedding_dim % 2 == 0

    positions = positions.float().reshape(-1, 1)
    omega = torch.arange(
        embedding_dim // 2,
        device=positions.device,
        dtype=torch.float32,
    )
    omega = omega / (embedding_dim // 2)
    omega = 1.0 / (10000 ** omega)
    angles = positions * omega[None]

    return torch.cat(
        [angles.sin(), angles.cos()],
        dim=-1,
    )


def sincos_grid_2d(
    height,
    width,
    hidden_dim,
    device,
):
    assert hidden_dim % 4 == 0

    grid_y, grid_x = torch.meshgrid(
        torch.arange(height, device=device),
        torch.arange(width, device=device),
        indexing="ij",
    )

    return torch.cat(
        [
            sinusoidal_1d(
                grid_y.reshape(-1),
                hidden_dim // 2,
            ),
            sinusoidal_1d(
                grid_x.reshape(-1),
                hidden_dim // 2,
            ),
        ],
        dim=-1,
    )


def sincos_grid_3d(
    depth,
    height,
    width,
    hidden_dim,
    device,
):
    assert hidden_dim % 6 == 0

    grid_z, grid_y, grid_x = torch.meshgrid(
        torch.arange(depth, device=device),
        torch.arange(height, device=device),
        torch.arange(width, device=device),
        indexing="ij",
    )
    axis_dim = hidden_dim // 3

    return torch.cat(
        [
            sinusoidal_1d(grid_z.reshape(-1), axis_dim),
            sinusoidal_1d(grid_y.reshape(-1), axis_dim),
            sinusoidal_1d(grid_x.reshape(-1), axis_dim),
        ],
        dim=-1,
    )


class TimestepEmbedder(nn.Module):
    def __init__(
        self,
        hidden_dim,
        frequency_dim=32,
    ):
        super().__init__()
        self.frequency_dim = frequency_dim
        self.mlp = nn.Sequential(
            nn.Linear(frequency_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, t):
        frequency_embedding = sinusoidal_1d(
            t * 1000.0,
            self.frequency_dim,
        )
        return self.mlp(frequency_embedding)


def modulate(x, shift, scale):
    return x * (1 + scale[:, None]) + shift[:, None]


class DiTBlock(nn.Module):
    def __init__(
        self,
        hidden_dim,
        num_heads,
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            hidden_dim,
            elementwise_affine=False,
            eps=1e-6,
        )
        self.attention = SelfAttention(
            hidden_dim,
            num_heads,
        )
        self.norm2 = nn.LayerNorm(
            hidden_dim,
            elementwise_affine=False,
            eps=1e-6,
        )
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, 4 * hidden_dim),
            nn.GELU(approximate="tanh"),
            nn.Linear(4 * hidden_dim, hidden_dim),
        )
        self.ada_ln_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_dim, 6 * hidden_dim),
        )

        nn.init.zeros_(self.ada_ln_modulation[-1].weight)
        nn.init.zeros_(self.ada_ln_modulation[-1].bias)

    def forward(
        self,
        x,
        condition,
        return_attn=False,
    ):
        (
            shift_attn,
            scale_attn,
            gate_attn,
            shift_mlp,
            scale_mlp,
            gate_mlp,
        ) = self.ada_ln_modulation(condition).chunk(6, dim=-1)

        attention_input = modulate(
            self.norm1(x),
            shift_attn,
            scale_attn,
        )
        attention_output, weights = self.attention(attention_input)
        x = x + gate_attn[:, None] * attention_output

        mlp_input = modulate(
            self.norm2(x),
            shift_mlp,
            scale_mlp,
        )
        mlp_output = self.mlp(mlp_input)
        x = x + gate_mlp[:, None] * mlp_output

        if return_attn:
            return x, weights
        return x


class DiTFinalLayer(nn.Module):
    def __init__(
        self,
        hidden_dim,
        output_dim,
    ):
        super().__init__()

        self.norm = nn.LayerNorm(
            hidden_dim,
            elementwise_affine=False,
            eps=1e-6,
        )
        self.ada_ln_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_dim, 2 * hidden_dim),
        )
        self.output_projection = nn.Linear(
            hidden_dim,
            output_dim,
        )

        nn.init.zeros_(self.ada_ln_modulation[-1].weight)
        nn.init.zeros_(self.ada_ln_modulation[-1].bias)
        nn.init.zeros_(self.output_projection.weight)
        nn.init.zeros_(self.output_projection.bias)

    def forward(
        self,
        x,
        condition,
    ):
        shift, scale = self.ada_ln_modulation(condition).chunk(2, dim=-1)
        x = modulate(
            self.norm(x),
            shift,
            scale,
        )
        return self.output_projection(x)

In [ ]:
class Tiny2DDiT(nn.Module):
    def __init__(
        self,
        channels=2,
        image_size=8,
        patch_size=2,
        hidden_dim=24,
        num_heads=3,
        depth=2,
    ):
        super().__init__()

        self.channels = channels
        self.image_size = image_size
        self.patch_size = patch_size
        self.hidden_dim = hidden_dim
        patch_dim = channels * patch_size * patch_size

        self.input_projection = nn.Linear(
            patch_dim,
            hidden_dim,
        )
        self.time_embedding = TimestepEmbedder(hidden_dim)
        self.blocks = nn.ModuleList(
            [
                DiTBlock(hidden_dim, num_heads)
                for _ in range(depth)
            ]
        )
        self.final_layer = DiTFinalLayer(
            hidden_dim,
            patch_dim,
        )

    def forward(
        self,
        x,
        t,
        return_attn=False,
    ):
        patches = F.unfold(
            x,
            kernel_size=self.patch_size,
            stride=self.patch_size,
        ).transpose(1, 2)

        grid_size = self.image_size // self.patch_size
        position_embedding = sincos_grid_2d(
            grid_size,
            grid_size,
            self.hidden_dim,
            x.device,
        ).to(x.dtype)

        hidden = self.input_projection(patches)
        hidden = hidden + position_embedding[None]
        condition = self.time_embedding(t.reshape(-1))

        attention_maps = []
        for block in self.blocks:
            if return_attn:
                hidden, weights = block(
                    hidden,
                    condition,
                    return_attn=True,
                )
                attention_maps.append(weights)
            else:
                hidden = block(hidden, condition)

        output_patches = self.final_layer(
            hidden,
            condition,
        ).transpose(1, 2)

        output = F.fold(
            output_patches,
            output_size=(self.image_size, self.image_size),
            kernel_size=self.patch_size,
            stride=self.patch_size,
        )

        if return_attn:
            return output, attention_maps
        return output


dit2 = Tiny2DDiT().to(device)
x_data = torch.randn(3, 2, 8, 8, device=device)
x_noise = torch.randn_like(x_data)
optimizer = torch.optim.AdamW(
    dit2.parameters(),
    lr=3e-3,
)

for step in range(6):
    t = torch.rand(3, device=device)
    t_broadcast = t[:, None, None, None]
    x_t = (
        (1 - t_broadcast) * x_data
        + t_broadcast * x_noise
    )
    target_velocity = x_noise - x_data
    predicted_velocity = dit2(x_t, t)
    loss = F.mse_loss(
        predicted_velocity,
        target_velocity,
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(
        "2D DiT-flow train",
        step,
        "loss",
        round(loss.item(), 4),
    )


@torch.no_grad()
def sample_2d_flow(
    model,
    initial_noise,
    num_steps=6,
):
    sample = initial_noise.clone()
    trajectory = [sample.clone()]
    dt = 1.0 / num_steps

    for step in range(num_steps):
        t_value = 1.0 - step * dt
        t = torch.full(
            (sample.size(0),),
            t_value,
            device=sample.device,
        )
        velocity = model(sample, t)
        sample = sample - dt * velocity
        trajectory.append(sample.clone())

    return sample, trajectory


sample_2d, trajectory_2d = sample_2d_flow(
    dit2,
    x_noise[:1],
)
print(
    "2D sample shape:",
    tuple(sample_2d.shape),
)
for index, state in enumerate(trajectory_2d):
    print(
        "2D sample",
        index,
        "mean",
        round(state.mean().item(), 4),
    )

In [ ]:
def patchify_3d(x, patch_size):
    (
        batch_size,
        channels,
        depth,
        height,
        width,
    ) = x.shape

    assert depth % patch_size == 0
    assert height % patch_size == 0
    assert width % patch_size == 0

    grid_depth = depth // patch_size
    grid_height = height // patch_size
    grid_width = width // patch_size

    x = x.reshape(
        batch_size,
        channels,
        grid_depth,
        patch_size,
        grid_height,
        patch_size,
        grid_width,
        patch_size,
    )
    x = x.permute(
        0, 2, 4, 6, 1, 3, 5, 7
    ).contiguous()

    num_patches = grid_depth * grid_height * grid_width
    patch_dim = channels * patch_size**3

    return x.reshape(
        batch_size,
        num_patches,
        patch_dim,
    )


def unpatchify_3d(
    tokens,
    channels,
    volume_size,
    patch_size,
):
    batch_size = tokens.size(0)
    grid_size = volume_size // patch_size

    x = tokens.reshape(
        batch_size,
        grid_size,
        grid_size,
        grid_size,
        channels,
        patch_size,
        patch_size,
        patch_size,
    )
    x = x.permute(
        0, 4, 1, 5, 2, 6, 3, 7
    ).contiguous()

    return x.reshape(
        batch_size,
        channels,
        volume_size,
        volume_size,
        volume_size,
    )


class Tiny3DDiT(nn.Module):
    def __init__(
        self,
        channels=1,
        volume_size=4,
        patch_size=2,
        hidden_dim=24,
        num_heads=3,
        depth=2,
    ):
        super().__init__()

        self.channels = channels
        self.volume_size = volume_size
        self.patch_size = patch_size
        self.hidden_dim = hidden_dim
        patch_dim = channels * patch_size**3

        self.input_projection = nn.Linear(
            patch_dim,
            hidden_dim,
        )
        self.time_embedding = TimestepEmbedder(hidden_dim)
        self.blocks = nn.ModuleList(
            [
                DiTBlock(hidden_dim, num_heads)
                for _ in range(depth)
            ]
        )
        self.final_layer = DiTFinalLayer(
            hidden_dim,
            patch_dim,
        )

    def forward(self, x, t):
        grid_size = self.volume_size // self.patch_size
        patches = patchify_3d(
            x,
            self.patch_size,
        )
        position_embedding = sincos_grid_3d(
            grid_size,
            grid_size,
            grid_size,
            self.hidden_dim,
            x.device,
        ).to(x.dtype)

        hidden = self.input_projection(patches)
        hidden = hidden + position_embedding[None]
        condition = self.time_embedding(t.reshape(-1))

        for block in self.blocks:
            hidden = block(hidden, condition)

        output_patches = self.final_layer(
            hidden,
            condition,
        )
        return unpatchify_3d(
            output_patches,
            self.channels,
            self.volume_size,
            self.patch_size,
        )


dit3 = Tiny3DDiT().to(device)
volume_data = torch.randn(3, 1, 4, 4, 4, device=device)
volume_noise = torch.randn_like(volume_data)
optimizer = torch.optim.AdamW(
    dit3.parameters(),
    lr=3e-3,
)

for step in range(6):
    t = torch.rand(3, device=device)
    t_broadcast = t[:, None, None, None, None]
    volume_t = (
        (1 - t_broadcast) * volume_data
        + t_broadcast * volume_noise
    )
    target_velocity = volume_noise - volume_data
    predicted_velocity = dit3(volume_t, t)
    loss = F.mse_loss(
        predicted_velocity,
        target_velocity,
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(
        "3D DiT-flow train",
        step,
        "loss",
        round(loss.item(), 4),
    )

with torch.no_grad():
    sample = volume_noise[:1].clone()
    num_steps = 6
    dt = 1.0 / num_steps

    for step in range(num_steps):
        t_value = 1.0 - step * dt
        t = torch.full((1,), t_value, device=device)
        velocity = dit3(sample, t)
        sample = sample - dt * velocity
        print(
            "3D sample",
            step,
            "mean",
            round(sample.mean().item(), 4),
        )

## E. Tiny π0-like VLA Flow Policy

vision+language는 prefix, robot state+noisy action horizon은 suffix로 두고 **같은 masked Transformer attention**에 넣는다. 학습 뒤에는 noise action chunk에서 시작해 velocity를 여러 step 적분해 continuous action chunk를 만든다.

In [ ]:
def make_pi0_like_mask(
    prefix_length,
    state_length,
    action_length,
    device,
):
    total_length = (
        prefix_length
        + state_length
        + action_length
    )
    allowed = torch.zeros(
        total_length,
        total_length,
        dtype=torch.bool,
        device=device,
    )

    allowed[:prefix_length, :prefix_length] = True

    state_start = prefix_length
    state_end = state_start + state_length
    allowed[state_start:state_end, :state_end] = True

    action_start = state_end
    allowed[action_start:, :] = True

    return allowed


class TinyVLAFlowPolicy(nn.Module):
    def __init__(
        self,
        patch_size=8,
        vocab_size=32,
        hidden_dim=24,
        num_heads=3,
        depth=2,
        action_dim=4,
        action_horizon=4,
    ):
        super().__init__()

        self.patch_size = patch_size
        self.action_horizon = action_horizon

        self.vision_projection = nn.Linear(
            3 * patch_size * patch_size,
            hidden_dim,
        )
        self.language_embedding = nn.Embedding(
            vocab_size,
            hidden_dim,
        )
        self.state_projection = nn.Linear(
            action_dim,
            hidden_dim,
        )
        self.action_input_projection = nn.Linear(
            action_dim,
            hidden_dim,
        )
        self.time_embedding = TimestepEmbedder(hidden_dim)
        self.blocks = nn.ModuleList(
            [
                PreNormBlock(
                    hidden_dim,
                    num_heads,
                    use_rope=True,
                )
                for _ in range(depth)
            ]
        )
        self.final_norm = nn.LayerNorm(hidden_dim)
        self.action_output_projection = nn.Linear(
            hidden_dim,
            action_dim,
        )

    def forward(
        self,
        image,
        language_ids,
        state,
        noisy_actions,
        t,
        return_attn=False,
    ):
        image_patches = F.unfold(
            image,
            kernel_size=self.patch_size,
            stride=self.patch_size,
        ).transpose(1, 2)
        image_tokens = self.vision_projection(image_patches)
        language_tokens = self.language_embedding(language_ids)
        prefix = torch.cat(
            [image_tokens, language_tokens],
            dim=1,
        )

        state_token = self.state_projection(state).unsqueeze(1)
        action_tokens = self.action_input_projection(noisy_actions)
        time_tokens = self.time_embedding(t.reshape(-1))[:, None]
        action_tokens = action_tokens + time_tokens

        x = torch.cat(
            [prefix, state_token, action_tokens],
            dim=1,
        )
        allowed_mask = make_pi0_like_mask(
            prefix_length=prefix.size(1),
            state_length=1,
            action_length=self.action_horizon,
            device=x.device,
        )

        attention_maps = []
        for block in self.blocks:
            x, weights = block(
                x,
                allowed_mask,
            )
            attention_maps.append(weights)

        action_hidden = x[:, -self.action_horizon :]
        action_hidden = self.final_norm(action_hidden)
        velocity = self.action_output_projection(action_hidden)

        if return_attn:
            return velocity, attention_maps, allowed_mask
        return velocity


policy = TinyVLAFlowPolicy().to(device)
batch_size = 3
vision = torch.randn(
    batch_size, 3, 16, 16,
    device=device,
)
language = torch.tensor(
    [
        [1, 2, 3],
        [4, 5, 6],
        [7, 8, 9],
    ],
    device=device,
)
robot_state = torch.randn(
    batch_size, 4,
    device=device,
)
action_data = torch.randn(
    batch_size, 4, 4,
    device=device,
)
action_noise = torch.randn_like(action_data)
optimizer = torch.optim.AdamW(
    policy.parameters(),
    lr=3e-3,
)

for step in range(6):
    t = torch.rand(batch_size, device=device)
    t_broadcast = t[:, None, None]
    action_t = (
        (1 - t_broadcast) * action_data
        + t_broadcast * action_noise
    )
    target_velocity = action_noise - action_data
    predicted_velocity = policy(
        vision,
        language,
        robot_state,
        action_t,
        t,
    )
    loss = F.mse_loss(
        predicted_velocity,
        target_velocity,
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(
        "VLA-flow train",
        step,
        "loss",
        round(loss.item(), 4),
    )


@torch.no_grad()
def sample_action_flow(
    model,
    image,
    language_ids,
    state,
    initial_noise,
    num_steps=6,
):
    actions = initial_noise.clone()
    trajectory = [actions.clone()]
    dt = 1.0 / num_steps

    for step in range(num_steps):
        t_value = 1.0 - step * dt
        t = torch.full(
            (actions.size(0),),
            t_value,
            device=actions.device,
        )
        velocity = model(
            image,
            language_ids,
            state,
            actions,
            t,
        )
        actions = actions - dt * velocity
        trajectory.append(actions.clone())

    return actions, trajectory


sampled_actions, action_trajectory = sample_action_flow(
    policy,
    vision[:1],
    language[:1],
    robot_state[:1],
    action_noise[:1],
)
print(
    "sampled action chunk:",
    sampled_actions,
)

with torch.no_grad():
    velocity, attention_maps, allowed_mask = policy(
        vision[:1],
        language[:1],
        robot_state[:1],
        action_noise[:1],
        torch.ones(1, device=device),
        return_attn=True,
    )

print(
    "joint attention shape:",
    tuple(attention_maps[0].shape),
)
print(
    "prefix -> action:",
    bool(allowed_mask[0, -1]),
)
print(
    "action -> prefix:",
    bool(allowed_mask[-1, 0]),
)

## References and provenance

- **GPT / Transformer** — Vaswani et al., *Attention Is All You Need*; Radford et al., GPT/GPT-2.
- **ViT** — Dosovitskiy et al., *An Image is Worth 16x16 Words*.
- **DiT** — Peebles & Xie, *Scalable Diffusion Models with Transformers* 및 공식 DiT 구현의 adaLN-Zero 구조.
- **Flow Matching** — Lipman et al., *Flow Matching for Generative Modeling*.
- **VLA / π0** — Physical Intelligence, *π0: A Vision-Language-Action Flow Model for General Robot Control* 및 openpi의 prefix/suffix attention 구조.

이 노트북은 외부 helper `.py` 없이 **모델 정의부터 학습과 추론까지 모두 이 파일 안에서 보이도록 구성**했다.